# Verificación de calidad (tarea final del job)

Última tarea del job: comprueba que las tablas que acaban de producir los cuadernos 01 y 02 son coherentes. Si alguna comprobación falla, **lanza una excepción**: la tarea queda en rojo, el job se marca como fallido y se dispara la notificación configurada. Así un problema de datos no llega en silencio a los informes.

| Comprobación | Por qué |
|---|---|
| Sin claves nulas ni duplicadas en plata | La clave identifica cada siniestro |
| Cuadre bronce = plata + cuarentena | Ninguna fila se perdió ni se duplicó por el camino |
| Cuarentena por debajo del umbral | Si el origen empeora de golpe, mejor parar y avisar |
| Ninguna regla crítica incumplida | Las reglas con acción *fallar* |
| Oro cuadra con plata | El resumen suma lo mismo que el detalle |
| Llamadas sin duraciones negativas en plata | La limpieza del cuaderno 01 funcionó |

In [1]:
import os

EN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if EN_DATABRICKS:
    # En Databricks: catálogo y esquema llegan como parámetros (widgets / parámetros del job)
    dbutils.widgets.text('catalogo', 'adb-dp750')
    dbutils.widgets.text('esquema', 'curso09')
    CATALOGO = dbutils.widgets.get('catalogo')
    ESQUEMA = dbutils.widgets.get('esquema')
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{CATALOGO}`.{ESQUEMA}')
    spark.sql(f'CREATE VOLUME IF NOT EXISTS `{CATALOGO}`.{ESQUEMA}.datos_crudos')
    RUTA_DATOS = f'/Volumes/{CATALOGO}/{ESQUEMA}/datos_crudos'
    PREFIJO = f'`{CATALOGO}`.{ESQUEMA}'

    # Si el CSV aún no está en el volumen, se descarga del repositorio
    import urllib.request
    for archivo in []:
        destino = f'{RUTA_DATOS}/{archivo}'
        if not os.path.exists(destino):
            urllib.request.urlretrieve('https://raw.githubusercontent.com/MiguelAP30/learning-data-science/main/cursos/09-databricks/datasets/' + archivo, destino)
else:
    # Fuera de Databricks: Spark + Delta Lake en local (ver entorno_local.py)
    from entorno_local import sesion_spark_local
    spark = sesion_spark_local('curso09-verificacion')
    ESQUEMA = 'curso09'
    spark.sql(f'CREATE DATABASE IF NOT EXISTS {ESQUEMA}')
    RUTA_DATOS = '../datasets'
    PREFIJO = ESQUEMA


def tabla(nombre):
    """Nombre completo de una tabla: catálogo.esquema.tabla en Databricks, esquema.tabla en local."""
    return f'{PREFIJO}.{nombre}'


print(f'Entorno: {"Databricks" if EN_DATABRICKS else "local"} · Spark {spark.version} · tablas en {PREFIJO}')

Entorno: local · Spark 4.0.1 · tablas en curso09


In [2]:
from pyspark.sql import functions as F

UMBRAL_CUARENTENA_PCT = 50

plata = spark.table(tabla('plata_siniestros'))
bronce = spark.table(tabla('bronce_siniestros'))
ejecucion = bronce.select('_id_ejecucion').first()[0]
metricas = spark.table(tabla('calidad_metricas')).filter(F.col('id_ejecucion') == ejecucion)
cuarentena = spark.table(tabla('cuarentena_siniestros')).filter(F.col('_id_ejecucion') == ejecucion)

n_bronce, n_cuarentena = bronce.count(), cuarentena.count()
n_plata_lote = plata.filter(F.col('_id_ejecucion') == ejecucion).count()
pct_cuarentena = round(n_cuarentena / n_bronce * 100, 1)
suma_plata = plata.agg(F.sum('claim_amount')).first()[0]
suma_oro = spark.table(tabla('oro_resumen_siniestros')).agg(F.sum('importe_total')).first()[0]
criticas = metricas.filter("accion = 'fallar'").agg(F.sum('filas_incumplen')).first()[0] or 0

comprobaciones = [
    ('Claves no nulas en plata', plata.filter('claim_id IS NULL').count() == 0, ''),
    ('Sin claim_id duplicados en plata', plata.count() == plata.select('claim_id').distinct().count(), ''),
    ('Cuadre bronce = plata + cuarentena', n_bronce == n_plata_lote + n_cuarentena,
     f'{n_bronce} = {n_plata_lote} + {n_cuarentena}'),
    (f'Cuarentena ≤ {UMBRAL_CUARENTENA_PCT}%', pct_cuarentena <= UMBRAL_CUARENTENA_PCT, f'{pct_cuarentena}%'),
    ('Reglas críticas sin incumplir', criticas == 0, f'{criticas} filas'),
    ('Oro cuadra con plata', suma_plata == suma_oro, f'{suma_plata} = {suma_oro}'),
    ('Llamadas sin duración negativa', spark.table(tabla('plata_llamadas')).filter('duration_seconds < 0').count() == 0, ''),
]

for nombre, ok, detalle in comprobaciones:
    print(f"{'[OK]   ' if ok else '[FALLO]'} {nombre:40} {detalle}")

fallidas = [nombre for nombre, ok, _ in comprobaciones if not ok]
if fallidas:
    raise AssertionError(f'Verificación de calidad fallida: {fallidas}')
print(f'\nEjecución {ejecucion}: todas las comprobaciones superadas')

[OK]    Claves no nulas en plata                 
[OK]    Sin claim_id duplicados en plata         
[OK]    Cuadre bronce = plata + cuarentena       25 = 16 + 9
[OK]    Cuarentena ≤ 50%                         36.0%
[OK]    Reglas críticas sin incumplir            0 filas
[OK]    Oro cuadra con plata                     243930.00 = 243930.00
[OK]    Llamadas sin duración negativa           

Ejecución 89543031: todas las comprobaciones superadas
